# Préparation de l'espace de travail

In [218]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

In [219]:
data_licences = pd.read_parquet("data/data_licences/data_licences.parquet")

# Graphiques

In [220]:
def plot_licencies_age_plotly(df, age):
    """
    Renvoie un graphique d'évolution des effectifs de licenciés par âge et par sport.
    
    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    age (str) : âge sélectionné
    
    Output
    ---------
    graphique
    """
    
    df_clean = df[df["code_sport"] != "DIV"]
    
    if age != "all":
        df_filtre = df_clean[df_clean["age"] == age]
        titre_age = f"{age} ans"
    else:
        df_filtre = df_clean.copy()
        titre_age = "tous les âges"

    table = (
        df_filtre.groupby(["annee", "code_sport"])["licences_annuelles"]
        .sum()
        .unstack()
        .sort_index()
    )

    table_long = table.reset_index().melt(
        id_vars="annee",
        var_name="code_sport",
        value_name="licences_annuelles"
    )

    # Tracé
    fig = px.line(
        table_long,
        x="annee",
        y="licences_annuelles",
        color="code_sport",
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        markers=True,
        labels={
            "annee": "Année",
            "licences_annuelles": "Nombre de licenciés",
            "code_sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés de {titre_age} par sport"
    )

    fig.update_layout(width=1100, height=600)
    fig.show()

In [221]:
# Code interactif

# --- Préparer les options pour les widgets ---
# Séparer les âges numériques et les autres
ages_numeric = sorted([a for a in data_licences["age"].dropna().unique() if a not in ["NR - Non réparti"]], key=int)

# Ajouter NR à la fin si présent
ages = ages_numeric
if "NR - Non réparti" in data_licences["age"].unique():
    ages.append("NR - Non réparti")

# Créer le widget
age_widget = widgets.Dropdown(options=["all"] + ages, description="Age :", value="all")

out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(age_widget)
    plot_licencies_age_plotly(data_licences, age_widget.value)

# --- Lier les widgets ---
age_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(age_widget, out)
update_graph()


Dropdown(description='Age :', options=('all', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '…

In [222]:
def plot_licencies_age_tranche_fine_plotly(df, tranche):
    """ Renvoie un graphique d'évolution des effectifs de licenciés par tranche d'âge et par sport.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    tranche (str) : tranche d'âge sélectionnée, indexée par des lettres (par exemple "a" pour 1 à 4 ans)
    
    Output
    ---------
    graphique
    """

    df_clean = df[df["code_sport"] != "DIV"]

    if tranche != "all":
        df_filtre = df_clean[df_clean["tranche_age"] == tranche]
        titre_age = f"tranche {tranche}"
    else:
        df_filtre = df_clean.copy()
        titre_age = "toutes les tranches"

    # S'assurer que l'année est triée
    df_filtre = df_filtre.sort_values(["annee", "code_sport"])

    # Récupérer la tranche lisible
    try:
        tranche_label = df_filtre["tranche_age"].str[4:].unique()[0]
    except:
        tranche_label = ""

    # Aggrégation
    table = (
        df_filtre.groupby(["annee", "code_sport"])["licences_annuelles"]
          .sum()
          .unstack()
          .sort_index()
    )

    table_long = table.reset_index().melt(
        id_vars="annee",
        var_name="code_sport",
        value_name="licences_annuelles"
    )

    # Tracé
    fig = px.line(
        table_long,
        x="annee",
        y="licences_annuelles",
        color="code_sport",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        labels={
            "annee": "Année",
            "licences_annuelles": "Nombre de licenciés",
            "code_sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés {titre_age} ans par sport"
    )

    fig.update_layout(width=1200, height=650)
    fig.show()



In [223]:
# Code interactif

# Créer le widget
ages = sorted(data_licences["tranche_age"].dropna().unique())

# Ajouter l'option "all" pour toutes les tranches
options = ["all"] + list(ages)

age_widget = widgets.Dropdown(
    options=options,
    description="Tranche :",
    value="all"
)

out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(age_widget)
    plot_licencies_age_tranche_fine_plotly(data_licences, age_widget.value)

# --- Lier les widgets ---
age_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(age_widget, out)
update_graph()


Dropdown(description='Tranche :', options=('all', 'NR - Non réparti', 'a - de 1 à 4 ans', 'b - de 5 à 9 ans', …

In [242]:
def sport_age_decomposition_plotly(df, annee):
    """
    Renvoie un graphique décomposant les effectifs de licenciés par tranche d'âge (grande tranche) et par sport pour une année.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    annee (int) : année sélectionnée
    
    Output
    ---------
    graphique
    """

     # --- Filtrer selon l'année ---
    if annee == "all":
        df_clean = df.copy()
        titre_annee = "2016 - 2024"
    else:
        df_clean = df[df["annee"] == annee]
        titre_annee = annee

    df_pivot = df_clean.pivot_table(
        index='code_sport',
        columns='grande_tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0)

    df_long = df_prop.reset_index().melt(
        id_vars="code_sport",
        var_name="grande_tranche_age",
        value_name="proportion"
    )

    # Passer en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100

    # --- Construire le mapping des couleurs ---
    tranches = sorted(df_long["grande_tranche_age"].unique())  # liste des tranches d'âge

    # Si "NR" existe, on l'isole
    palette_map = {}

    if "NR - Non réparti" in tranches:
        tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]
    else:
        tranches_no_nr = tranches

    # Nombre de couleurs à générer pour les tranches (hors NR)
    n = len(tranches_no_nr)

    # Générer n couleurs dans Plasma_r, du jaune (min) au violet (max)
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n > 1 else [0.5]
    )

    # Assigner les couleurs aux tranches (hors NR)
    for tranche, col in zip(tranches_no_nr, colors):
        palette_map[tranche] = col

    # Assigner NR en noir si présent
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"

    # --- Graphique ---
    fig = px.bar(
        df_long,
        x="proportion",
        y="code_sport",
        color="grande_tranche_age",
        color_discrete_map=palette_map,
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "code_sport": "Sport",
            "grande_tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et tranche d'âge – {titre_annee}"
    )

    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=600,
        xaxis=dict(range=[0, 100])
    )

    fig.show()


In [243]:
# Code interactif

# Créer le widget
annees = sorted(data_licences["annee"].dropna().unique())

# Ajouter l'option "all" pour toutes les tranches
options = ["all"] + list(annees)

annees_widget = widgets.Dropdown(
    options=options,
    description="Années :",
    value="all"
)

out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(annees_widget)
    sport_age_decomposition_plotly(data_licences, annees_widget.value)

# --- Lier les widgets ---
annees_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(annees_widget, out)
update_graph()


Dropdown(description='Années :', options=('all', np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019…

In [248]:
def sport_age_decomposition_fine_plotly(df, annee):
    """ Renvoie un graphique décomposant les effectifs de licenciés par tranche d'âge (tranche d'âge fine) et par sport pour une année.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    annee (int) : année sélectionnée
    
    Output
    ---------
    graphique
    """

    if annee == "all":
            df_clean = df.copy()
            titre_annee = "2016 - 2024"
    else:
        df_clean = df[df["annee"] == annee]
        titre_annee = annee

    df_pivot = df_clean.pivot_table(
        index='code_sport',
        columns='tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    # Proportion par ligne
    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0).fillna(0)

    # Format long
    df_long = df_prop.reset_index().melt(
        id_vars="code_sport",
        var_name="tranche_age",
        value_name="proportion"
    )

    # Supprimer tranches vides ou NaN
    df_long = df_long[df_long["tranche_age"].notna()]
    df_long["tranche_age"] = df_long["tranche_age"].astype(str)

    # Passer en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100


    # Couleurs
    tranches = sorted(df_long["tranche_age"].unique())
    
    # Séparer NR
    tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]

    n = len(tranches_no_nr)

    # Générer n couleurs dans Plasma_r
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n>1 else [0.5]
    )

    palette_map = {t: c for t, c in zip(tranches_no_nr, colors)}
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"
    

    # --- Définir l'ordre des tranches ---
    # Trier alphabétiquement sauf NR
    tranches_ord = sorted(tranches_no_nr)
    # Ajouter NR à la fin si présent
    if "NR - Non réparti" in tranches:
        tranches_ord.append("NR - Non réparti")


    # Tracé
    fig = px.bar(
        df_long,
        x="proportion",
        y="code_sport",
        color="tranche_age",
        color_discrete_map=palette_map,
        category_orders={"tranche_age": tranches_ord},
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "code_sport": "Sport",
            "tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et tranche d'âge – {titre_annee}"
    )

    # Axe X en pourcentage
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=600,
        xaxis=dict(range=[1, 100]))

    fig.show()

In [249]:
# Code interactif

# Créer le widget
annees = sorted(data_licences["annee"].dropna().unique())

# Ajouter l'option "all" pour toutes les tranches
options = ["all"] + list(annees)

annees_widget = widgets.Dropdown(
    options=options,
    description="Années :",
    value="all"
)

out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(annees_widget)
    sport_age_decomposition_fine_plotly(data_licences, annees_widget.value)

# --- Lier les widgets ---
annees_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(annees_widget, out)
update_graph()


Dropdown(description='Années :', index=6, options=('all', np.int64(2016), np.int64(2017), np.int64(2018), np.i…

In [236]:
# --- Préparer les options pour les widgets ---
ages = sorted(data_licences["tranche_age"].dropna().unique())
sports = sorted(data_licences["code_sport"].dropna().unique())
annees = sorted(data_licences["annee"].dropna().unique())

age_widget = widgets.Dropdown(options=ages, description="Âge :", value=ages[0])
sport_widget = widgets.Dropdown(options=["all"] + sports, description="Sport :", value="all")
annee_widget = widgets.Dropdown(options=["all"] + list(annees), description="Année :", value="all")

# --- Fonction pour filtrer et préparer les données ---
def filter_data(df, age, sport, annee):
    df_filtered = df[df["tranche_age"] == age]
    
    if sport != "all":
        df_filtered = df_filtered[df_filtered["code_sport"] == sport]
        
    if annee != "all":
        df_filtered = df_filtered[df_filtered["annee"] == annee]
    
    table = (
        df_filtered.groupby(["annee", "code_sport"])["licences_annuelles"]
        .sum()
        .reset_index()
    )
    return table

# --- Fonction pour tracer avec Plotly ---
def plot_licencies_age_interactive(age, sport, annee):
    df_plot = filter_data(data_licences, age, sport, annee)
    if df_plot.empty:
        fig = px.line(title=f"Aucune donnée pour l'âge {age} ans avec ce filtre")
    else:
        fig = px.line(
            df_plot,
            x="annee",
            y="licences_annuelles",
            color="code_sport",
            markers=True,
            title=f"Évolution des licenciés de {age} ans par sport"
        )
    fig.update_layout(
        xaxis_title="Année",
        yaxis_title="Nombre de licenciés",
        legend_title="Sport",
        template="plotly_white"
    )
    fig.show()

# --- Callback pour mettre à jour le graphique ---
def update_graph(change=None):
    clear_output(wait=True)
    display(age_widget, sport_widget, annee_widget)
    plot_licencies_age_interactive(age_widget.value, sport_widget.value, annee_widget.value)

# --- Lier les widgets ---
age_widget.observe(update_graph, names='value')
sport_widget.observe(update_graph, names='value')
annee_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(age_widget, sport_widget, annee_widget)
update_graph()


Dropdown(description='Âge :', options=('NR - Non réparti', 'a - de 1 à 4 ans', 'b - de 5 à 9 ans', 'c - de 10 …

Dropdown(description='Sport :', options=('all', 'ATH', 'AVI', 'BAD', 'BAK', 'BAS', 'BOX', 'CAK', 'CYC', 'DIV',…

Dropdown(description='Année :', options=('all', np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)…